In [ ]:
# Run this in first cell
from google.colab import files
uploaded = files.upload()
# A button appears → click → select your CSV file

Saving data.csv to data.csv


In [ ]:
!pip install delta-spark

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pyspark.sql.functions as F
from pyspark.sql.types import *

builder = SparkSession.builder \
    .appName("ecommerce_pipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Ready!", spark.version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.1 MB/s eta 0:00:00
Ready! 4.0.2


In [ ]:
# Define schema explicitly (never inferSchema in production!)
schema = StructType([
    StructField("InvoiceNo",   StringType(),  True),
    StructField("StockCode",   StringType(),  True),
    StructField("Description", StringType(),  True),
    StructField("Quantity",    IntegerType(), True),
    StructField("InvoiceDate", StringType(),  True),
    StructField("UnitPrice",   DoubleType(),  True),
    StructField("CustomerID",  StringType(),  True),
    StructField("Country",     StringType(),  True)
])

# Read CSV
df_raw = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("/content/data.csv")

print(f"Total rows: {df_raw.count()}")
df_raw.show(5)
df_raw.printSchema()

Total rows: 541909
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo: string (n

In [ ]:
# Add ingestion timestamp — standard Bronze practice
df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp())

# Write to Bronze Delta table
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/bronze/ecommerce")

print("Bronze layer written successfully!")
print(f"Bronze row count: {spark.read.format('delta').load('/content/delta/bronze/ecommerce').count()}")

Bronze layer written successfully!
Bronze row count: 541909


In [ ]:
# Read from Bronze
df_bronze = spark.read.format("delta").load("/content/delta/bronze/ecommerce")

df_silver = df_bronze \
    .drop("ingestion_timestamp") \
    .dropna(subset=["CustomerID", "InvoiceNo"]) \
    .filter(F.col("Quantity") > 0) \
    .filter(F.col("UnitPrice") > 0) \
    .dropDuplicates(["InvoiceNo", "StockCode"]) \
    .withColumn("InvoiceDate",
        F.to_timestamp(F.col("InvoiceDate"), "M/d/yyyy H:mm")) \
    .withColumn("TotalAmount",
        F.round(F.col("Quantity") * F.col("UnitPrice"), 2)) \
    .withColumn("Year",  F.year(F.col("InvoiceDate"))) \
    .withColumn("Month", F.month(F.col("InvoiceDate"))) \
    .withColumn("Date",  F.to_date(F.col("InvoiceDate")))

print(f"Bronze rows: 541909")
print(f"Silver rows: {df_silver.count()}")
df_silver.show(5)

Bronze rows: 541909
Silver rows: 387841
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalAmount|Year|Month|      Date|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+----------+
|   536365|    21730|GLASS STAR FROSTE...|       6|2010-12-01 08:26:00|     4.25|     17850|United Kingdom|       25.5|2010|   12|2010-12-01|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|      20.34|2010|   12|2010-12-01|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|       22.0|2010|   12|2010-12-01|
|   536366|    22632|HAND WARMER RED P...|       6|2010-12-01 08:28:00|     1.85|     17850|United Kingdom| 

In [ ]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("Year", "Month") \
    .save("/content/delta/silver/ecommerce")

print("Silver layer written successfully!")
print(f"Silver row count: {spark.read.format('delta').load('/content/delta/silver/ecommerce').count()}")

Silver layer written successfully!
Silver row count: 387841


In [ ]:
# Read from Silver
df_silver = spark.read.format("delta").load("/content/delta/silver/ecommerce")

# Daily revenue aggregation
df_daily_revenue = df_silver \
    .groupBy("Date", "Country") \
    .agg(
        F.sum("TotalAmount").alias("daily_revenue"),
        F.countDistinct("InvoiceNo").alias("total_orders"),
        F.countDistinct("CustomerID").alias("unique_customers"),
        F.sum("Quantity").alias("total_items_sold")
    ) \
    .orderBy("Date")

# Write to Gold
df_daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/gold/daily_revenue")

print("Gold Daily Revenue written!")
df_daily_revenue.show(5)

Gold Daily Revenue written!
+----------+--------------+-----------------+------------+----------------+----------------+
|      Date|       Country|    daily_revenue|total_orders|unique_customers|total_items_sold|
+----------+--------------+-----------------+------------+----------------+----------------+
|2010-12-01|        Norway|          1919.14|           1|               1|            1852|
|2010-12-01|          EIRE|555.3799999999999|           2|               1|             243|
|2010-12-01|United Kingdom|41756.79000000006|         114|              89|           21041|
|2010-12-01|     Australia|           358.25|           1|               1|             107|
|2010-12-01|   Netherlands|            192.6|           1|               1|              97|
+----------+--------------+-----------------+------------+----------------+----------------+
only showing top 5 rows


In [ ]:
df_top_products = df_silver \
    .groupBy("StockCode", "Description") \
    .agg(
        F.sum("TotalAmount").alias("total_revenue"),
        F.sum("Quantity").alias("total_quantity_sold"),
        F.countDistinct("CustomerID").alias("unique_customers"),
        F.countDistinct("InvoiceNo").alias("total_orders")
    ) \
    .orderBy(F.desc("total_revenue"))

# Write to Gold
df_top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/gold/top_products")

print("Gold Top Products written!")
df_top_products.show(5)

Gold Top Products written!
+---------+--------------------+------------------+-------------------+----------------+------------+
|StockCode|         Description|     total_revenue|total_quantity_sold|unique_customers|total_orders|
+---------+--------------------+------------------+-------------------+----------------+------------+
|    23843|PAPER CRAFT , LIT...|          168469.6|              80995|               1|           1|
|    22423|REGENCY CAKESTAND...|141945.99999999977|              12349|             881|        1703|
|   85123A|WHITE HANGING HEA...|100046.94999999976|              36589|             856|        1971|
|   85099B|JUMBO BAG RED RET...| 84962.28000000044|              46040|             635|        1600|
|    23166|MEDIUM CERAMIC TO...|          81405.48|              77907|             138|         195|
+---------+--------------------+------------------+-------------------+----------------+------------+
only showing top 5 rows


In [ ]:
# Calculate customer metrics
df_customer = df_silver \
    .groupBy("CustomerID", "Country") \
    .agg(
        F.sum("TotalAmount").alias("total_spent"),
        F.countDistinct("InvoiceNo").alias("total_orders"),
        F.min("Date").alias("first_purchase"),
        F.max("Date").alias("last_purchase"),
        F.sum("Quantity").alias("total_items")
    ) \
    .withColumn("avg_order_value",
        F.round(F.col("total_spent") / F.col("total_orders"), 2)) \
    .withColumn("customer_segment",
        F.when(F.col("total_spent") >= 10000, "VIP")
         .when(F.col("total_spent") >= 1000, "Regular")
         .otherwise("Occasional"))

# Write to Gold
df_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/gold/customer_segments")

print("Gold Customer Segments written!")
df_customer.show(5)

Gold Customer Segments written!
+----------+--------------+-----------------+------------+--------------+-------------+-----------+---------------+----------------+
|CustomerID|       Country|      total_spent|total_orders|first_purchase|last_purchase|total_items|avg_order_value|customer_segment|
+----------+--------------+-----------------+------------+--------------+-------------+-----------+---------------+----------------+
|     12600|       Germany|          2599.04|          10|    2010-12-03|   2011-11-11|        774|          259.9|         Regular|
|     13662|United Kingdom|           881.26|           3|    2011-03-25|   2011-09-15|        533|         293.75|      Occasional|
|     15861|United Kingdom|          3128.84|           8|    2011-03-17|   2011-11-27|       2500|         391.11|         Regular|
|     15374|United Kingdom|            168.0|           1|    2011-08-03|   2011-08-03|        126|          168.0|      Occasional|
|     14667|United Kingdom|9042.49999

In [ ]:
bronze = spark.read.format("delta").load("/content/delta/bronze/ecommerce").count()
silver = spark.read.format("delta").load("/content/delta/silver/ecommerce").count()
daily_rev = spark.read.format("delta").load("/content/delta/gold/daily_revenue").count()
top_prod = spark.read.format("delta").load("/content/delta/gold/top_products").count()
customers = spark.read.format("delta").load("/content/delta/gold/customer_segments").count()

print("=" * 40)
print("MEDALLION ARCHITECTURE SUMMARY")
print("=" * 40)
print(f"🥉 Bronze (raw):              {bronze:,} rows")
print(f"🥈 Silver (cleaned):          {silver:,} rows")
print(f"🥇 Gold - Daily Revenue:      {daily_rev:,} rows")
print(f"🥇 Gold - Top Products:       {top_prod:,} rows")
print(f"🥇 Gold - Customer Segments:  {customers:,} rows")
print("=" * 40)

MEDALLION ARCHITECTURE SUMMARY
🥉 Bronze (raw):              541,909 rows
🥈 Silver (cleaned):          387,841 rows
🥇 Gold - Daily Revenue:      1,523 rows
🥇 Gold - Top Products:       3,897 rows
🥇 Gold - Customer Segments:  4,346 rows


In [ ]:
from delta.tables import DeltaTable

# Simulate new incoming data (new day's transactions)
new_data = [
    ("999999", "85123A", "WHITE HANGING HEART", 10, "2011-12-10", 2.55, "17850", "United Kingdom", None),
    ("999998", "71053",  "WHITE METAL LANTERN", 5,  "2011-12-10", 3.39, "12345", "United Kingdom", None),
]

schema_new = StructType([
    StructField("InvoiceNo",   StringType(),  True),
    StructField("StockCode",   StringType(),  True),
    StructField("Description", StringType(),  True),
    StructField("Quantity",    IntegerType(), True),
    StructField("InvoiceDate", StringType(),  True),
    StructField("UnitPrice",   DoubleType(),  True),
    StructField("CustomerID",  StringType(),  True),
    StructField("Country",     StringType(),  True),
    StructField("ingestion_timestamp", StringType(), True)
])

df_new = spark.createDataFrame(new_data, schema_new) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

# MERGE into Bronze
delta_bronze = DeltaTable.forPath(spark, "/content/delta/bronze/ecommerce")

delta_bronze.alias("target").merge(
    df_new.alias("source"),
    "target.InvoiceNo = source.InvoiceNo AND target.StockCode = source.StockCode"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

print(f"After MERGE: {spark.read.format('delta').load('/content/delta/bronze/ecommerce').count():,} rows")

After MERGE: 541,911 rows


In [ ]:
# Check history
delta_bronze = DeltaTable.forPath(spark, "/content/delta/bronze/ecommerce")
delta_bronze.history().select("version","timestamp","operation").show()

# Query version 0 — before MERGE
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/content/delta/bronze/ecommerce")

print(f"Version 0 (original): {df_v0.count():,} rows")
print(f"Current version:      {spark.read.format('delta').load('/content/delta/bronze/ecommerce').count():,} rows")
print(f"Difference:           {spark.read.format('delta').load('/content/delta/bronze/ecommerce').count() - df_v0.count()} rows added by MERGE")

+-------+--------------------+---------+
|version|           timestamp|operation|
+-------+--------------------+---------+
|      1|2026-03-11 09:02:...|    MERGE|
|      0|2026-03-11 08:52:...|    WRITE|
+-------+--------------------+---------+

Version 0 (original): 541,909 rows
Current version:      541,911 rows
Difference:           2 rows added by MERGE


In [ ]:
# Save gold tables as CSV for Streamlit
spark.read.format("delta") \
    .load("/content/delta/gold/daily_revenue") \
    .toPandas() \
    .to_csv("/content/daily_revenue.csv", index=False)

spark.read.format("delta") \
    .load("/content/delta/gold/top_products") \
    .toPandas() \
    .to_csv("/content/top_products.csv", index=False)

spark.read.format("delta") \
    .load("/content/delta/gold/customer_segments") \
    .toPandas() \
    .to_csv("/content/customer_segments.csv", index=False)

print("Gold tables saved as CSV for dashboard!")

Gold tables saved as CSV for dashboard!


In [ ]:
!pip install streamlit plotly pyngrok -q
print("Streamlit installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.0 MB/s eta 0:00:00
Streamlit installed!


In [23]:
%%writefile /content/dashboard.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="E-Commerce Analytics", layout="wide")
st.title("E-Commerce Analytics Dashboard")
st.markdown("---")

daily_revenue = pd.read_csv("/content/daily_revenue.csv")
top_products  = pd.read_csv("/content/top_products.csv")
customers     = pd.read_csv("/content/customer_segments.csv")
daily_revenue["Date"] = pd.to_datetime(daily_revenue["Date"])

# KPI METRICS
st.header("Key Metrics")
col1, col2, col3, col4 = st.columns(4)
total_revenue   = daily_revenue["daily_revenue"].sum()
total_orders    = daily_revenue["total_orders"].sum()
total_customers = customers["CustomerID"].nunique()
avg_order_value = customers["avg_order_value"].mean()
col1.metric("Total Revenue",   f"GBP {total_revenue:,.0f}")
col2.metric("Total Orders",    f"{total_orders:,.0f}")
col3.metric("Total Customers", f"{total_customers:,.0f}")
col4.metric("Avg Order Value", f"GBP {avg_order_value:,.2f}")
st.markdown("---")

# DAILY REVENUE TREND
st.header("Daily Revenue Trend")
uk_revenue = daily_revenue[daily_revenue["Country"] == "United Kingdom"].groupby("Date")["daily_revenue"].sum().reset_index()
fig1 = px.line(uk_revenue, x="Date", y="daily_revenue",
    title="Daily Revenue United Kingdom",
    color_discrete_sequence=["#2196F3"])
st.plotly_chart(fig1, use_container_width=True)
st.markdown("---")

# TOP 10 PRODUCTS
st.header("Top 10 Products by Revenue")
top10 = top_products.head(10)
fig2 = px.bar(top10, x="total_revenue", y="Description",
    orientation="h",
    title="Top 10 Products by Revenue",
    color="total_revenue",
    color_continuous_scale="Blues")
fig2.update_layout(yaxis={"categoryorder": "total ascending"})
st.plotly_chart(fig2, use_container_width=True)
st.markdown("---")

# CUSTOMER SEGMENTATION
st.header("Customer Segmentation")
col1, col2 = st.columns(2)
segment_counts = customers["customer_segment"].value_counts().reset_index()
segment_counts.columns = ["segment", "count"]
fig3 = px.pie(segment_counts, values="count", names="segment",
    title="Customer Segments Distribution",
    color_discrete_sequence=["#4CAF50", "#2196F3", "#FF9800"])
col1.plotly_chart(fig3, use_container_width=True)
segment_revenue = customers.groupby("customer_segment")["total_spent"].sum().reset_index()
fig4 = px.bar(segment_revenue, x="customer_segment", y="total_spent",
    title="Revenue by Customer Segment",
    color="customer_segment",
    color_discrete_sequence=["#4CAF50", "#2196F3", "#FF9800"])
col2.plotly_chart(fig4, use_container_width=True)
st.markdown("---")

# REVENUE BY COUNTRY
st.header("Revenue by Country")
country_revenue = daily_revenue.groupby("Country")["daily_revenue"].sum().reset_index().sort_values("daily_revenue", ascending=False).head(10)
fig5 = px.bar(country_revenue, x="Country", y="daily_revenue",
    title="Top 10 Countries by Revenue",
    color="daily_revenue",
    color_continuous_scale="Blues")
st.plotly_chart(fig5, use_container_width=True)
st.markdown("---")
st.markdown("Built with PySpark + Delta Lake + Streamlit")

Overwriting /content/dashboard.py


In [24]:
from pyngrok import ngrok
import subprocess
import time

# Add your authtoken here
ngrok.set_auth_token("3AnGSG6L38dvLKv63rjUO0NMcrh_5aZ4ob2aGjKRSEiQTBBfp")  # ← paste your token

# Start streamlit
process = subprocess.Popen([
    "streamlit", "run", "/content/dashboard.py",
    "--server.port", "8501",
    "--server.headless", "true"
])

time.sleep(5)

# Create public URL
ngrok.kill()
public_url = ngrok.connect(8501)
print("=" * 50)
print("🚀 Dashboard is LIVE!")
print(f"👉 Open this URL: {public_url}")
print("=" * 50)

🚀 Dashboard is LIVE!
👉 Open this URL: NgrokTunnel: "https://jong-bowerlike-nigel.ngrok-free.dev" -> "http://localhost:8501"
